In [1]:
#get the time factor

def get_time_factor(days):
    if days < 0:
        raise ValueError('Day before dep cannot smaller than 0')
    if days <= 7:
        time_factor = 1.35
    elif days <= 21:
        time_factor = 1.1
    else:
        time_factor = 1
    return time_factor

print(get_time_factor(12))
try:
    get_time_factor(-1)
except ValueError:
    print('ilegal value has been stopped')
print(get_time_factor(0))



1.1
ilegal value has been stopped
1.35


In [2]:
class Flight:
    def __init__(self, flight_id, base_fare, seats_remaining, capacity, demand_factor, seasonal_factor, origin, destination):

        #Check case
        if capacity <= 0:
            raise ValueError('Capacity must be greater than 0')
        if not 0 <= seats_remaining <= capacity:
            raise ValueError('Seat remaining must within 0 and Capacity ')
        if base_fare <= 0:
            raise ValueError('Base fare must greater than 0')
        if demand_factor <= 0 or seasonal_factor <= 0:
            raise ValueError('Demand and seasonal factors must be greater than 0')
        
        self.flight_id = flight_id
        self.base_fare = base_fare
        self.seats_remaining = seats_remaining
        self.capacity = capacity
        self.demand_factor = demand_factor
        self.seasonal_factor = seasonal_factor
        self.origin = origin
        self.destination = destination

    def load_factor(self):
        return 1 - self.seats_remaining/self.capacity

    def update_seats(self, change):
        new_value = self.seats_remaining + change

        if not 0 <= new_value <= self.capacity:
            raise ValueError("seat update is outside 0..capacity")
        else:
            self.seats_remaining = new_value

    def calculate_price(self, days):
    #self or not: flight data - Y, tmp value - N
        time_factor = get_time_factor(days)
        capacity_factor = 1 + 0.45 * self.load_factor()
        price = self.base_fare * time_factor * capacity_factor * self.demand_factor * self.seasonal_factor
        return round(min(1500,max(45,price)), 2)


test_flight = Flight('PA1001', 300, 18, 180, 1.2, 1.0, 'YYZ', 'YVR')
print(test_flight.destination)


YVR


In [3]:
print(test_flight.seats_remaining)
test_flight.load_factor()
try:
    test_flight.update_seats(-20)
except ValueError:
    print("Invalid seat update blocked.")

18
Invalid seat update blocked.


In [4]:
old_price = test_flight.calculate_price(12)
test_flight.update_seats(-5)
new_price = test_flight.calculate_price(12)
print(old_price)
print(new_price)

556.38
561.33


In [5]:
try:
    bad_flight = Flight("PA_TEST", 300, 181, 180, 1.2, 1.0, 'YYZ', 'CHN')
except ValueError:
    print("Invalid flight update blocked.")

Invalid flight update blocked.


## SQLite Persistence

Run these cells in order from this notebook folder. The setup cell rebuilds the demo flights table from CSV, resetting previous database updates. Pricing integration is still pending.

### 1. Load CSV

In [6]:
# Load flight data.
import sqlite3
import pandas as pd

flights = pd.read_csv("flights.csv")


### 2. Create Table and Insert Flights

In [7]:
# Rebuild the demo table and import CSV records.
with sqlite3.connect("potter_airlines.db") as conn:
    conn.execute("DROP TABLE IF EXISTS flights")

    conn.execute("""
        CREATE TABLE flights (
            flight_id TEXT PRIMARY KEY,
            origin TEXT NOT NULL,
            destination TEXT NOT NULL,
            departure_date TEXT NOT NULL,
            base_fare REAL NOT NULL,
            capacity INTEGER NOT NULL,
            seats_remaining INTEGER NOT NULL,
            demand_factor REAL NOT NULL,
            seasonal_factor REAL NOT NULL
        )
    """)

    flight_records = flights.to_dict(orient="records")
    for flight_dict in flight_records:
        flight_values = (
            flight_dict["flight_id"], flight_dict["origin"],
            flight_dict["destination"], flight_dict["departure_date"],
            flight_dict["base_fare"], flight_dict["capacity"],
            flight_dict["seats_remaining"], flight_dict["demand_factor"],
            flight_dict["seasonal_factor"]
        )
        conn.execute(
            "INSERT INTO flights VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
            flight_values
        )
print("CSV flights inserted.")


CSV flights inserted.


### 3. Query Flights

In [8]:
# Query flights by destination.
destination = "YVR"
with sqlite3.connect("potter_airlines.db") as conn:
    rows = conn.execute(
        "SELECT * FROM flights WHERE destination = ?",
        (destination,)
    ).fetchall()
print("Flights to YVR:", rows)


Flights to YVR: [('PA1001', 'YYZ', 'YVR', '2026-09-17', 300.0, 180, 18, 1.2, 1.0), ('PA1002', 'YYZ', 'YVR', '2026-09-23', 300.0, 180, 90, 1.2, 1.0), ('PA1003', 'YYZ', 'YVR', '2026-10-07', 300.0, 180, 144, 1.2, 1.0), ('PA1004', 'YYZ', 'YVR', '2026-10-08', 300.0, 180, 144, 1.2, 1.0), ('PA1016', 'YYC', 'YVR', '2026-10-17', 180.0, 120, 36, 0.9, 1.0), ('PA1017', 'YYZ', 'YVR', '2026-12-20', 300.0, 180, 90, 1.2, 1.3)]


### 4. Update and Delete a Test Flight

In [9]:
# Demonstrate and verify seat updates and deletion on a test flight.
with sqlite3.connect("potter_airlines.db") as conn:
    conn.execute(
        "INSERT INTO flights VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
        ("SQL_DEMO", "YYZ", "YVR", "2026-10-01", 300, 180, 18, 1.2, 1.0)
    )

    conn.execute(
        "UPDATE flights SET seats_remaining = ? WHERE flight_id = ?",
        (13, "SQL_DEMO")
    )
    updated = conn.execute(
        "SELECT flight_id, seats_remaining FROM flights WHERE flight_id = ?",
        ("SQL_DEMO",)
    ).fetchall()
    print("Updated test flight:", updated)
    assert updated == [("SQL_DEMO", 13)]

    conn.execute(
        "DELETE FROM flights WHERE flight_id = ?",
        ("SQL_DEMO",)
    )
    deleted = conn.execute(
        "SELECT * FROM flights WHERE flight_id = ?",
        ("SQL_DEMO",)
    ).fetchall()
    assert deleted == []
    print("Test flight deleted:", deleted)


Updated test flight: [('SQL_DEMO', 13)]
Test flight deleted: []


### 5. Verify Stored Data

In [10]:
# Read persisted flights into a DataFrame and verify the row count.
with sqlite3.connect("potter_airlines.db") as conn:
    stored_flights = pd.read_sql_query("SELECT * FROM flights", conn)
assert len(stored_flights) == len(flights)
print("Stored flight count:", len(stored_flights))
print(stored_flights.head())

conn.close()


Stored flight count: 20
  flight_id origin destination  ... seats_remaining  demand_factor  seasonal_factor
0    PA1001    YYZ         YVR  ...              18            1.2              1.0
1    PA1002    YYZ         YVR  ...              90            1.2              1.0
2    PA1003    YYZ         YVR  ...             144            1.2              1.0
3    PA1004    YYZ         YVR  ...             144            1.2              1.0
4    PA1005    YVR         YYZ  ...               1            1.2              1.0

[5 rows x 9 columns]
